In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==============================================================================
# ПАРАМЕТРИ ВАРІАНТА 6 (м. Чернігів)
# ==============================================================================
VARIANT = 6
CITY = "Чернігів"
BASE_TEMP = 8.0
AMPLITUDE = 13.0
BASE_CONSUMPTION = 900
K = 25
NOISE_SD = 30

np.random.seed(VARIANT)

# Генерація вихідного набору даних
months = np.arange(1, 13)
temp = BASE_TEMP + AMPLITUDE * np.cos((months - 7) / 12 * 2 * np.pi)
temp = np.round(temp + np.random.normal(0, 0.5, size=12), 1)

# Споживання електроенергії (МВт·год)
consumption = BASE_CONSUMPTION - K * (temp - BASE_TEMP) + np.random.normal(0, NOISE_SD, size=12)
consumption = np.round(consumption, 0)

city_data = pd.DataFrame({
    "місяць": months,
    "температура": temp,
    "споживання_МВтгод": consumption,
})

print(f"=== ПРАКТИКА 19: Варіант {VARIANT} ({CITY}) ===")
print(city_data.to_string(index=False))

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 2. Побудова моделі на власних даних
# ------------------------------------------------------------------------------
print("\n=== Завдання 2: Побудова моделі ===")

b, a = np.polyfit(city_data["температура"], city_data["споживання_МВтгод"], 1)
print(f"Рівняння моделі: споживання ≈ {a:.1f} + ({b:.1f}) * температура")
print(f"Коефіцієнт нахилу b = {b:.2f} МВт·год/°C")
print(f"Вільний член a = {a:.2f} МВт·год")

city_data["передбачення"] = a + b * city_data["температура"]

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 3. Порівняння передбачень, залишки та R²
# ------------------------------------------------------------------------------
print("\n=== Завдання 3: Аналіз залишків та R² ===")

city_data["залишок"] = city_data["споживання_МВтгод"] - city_data["передбачення"]

# Таблиця порівняння
print(city_data[["місяць", "температура", "споживання_МВтгод", "передбачення", "залишок"]].to_string(index=False))

residuals = city_data["залишок"]
ss_res = (residuals ** 2).sum()
ss_tot = ((city_data["споживання_МВтгод"] - city_data["споживання_МВтгод"].mean()) ** 2).sum()
r_squared = 1 - ss_res / ss_tot

print(f"\nСума квадратів залишків (SS_res): {ss_res:.2f}")
print(f"Загальна сума квадратів (SS_tot): {ss_tot:.2f}")
print(f"Коефіцієнт детермінації R²:       {r_squared:.4f}")

# Побудова графіків
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Спостереження vs Модель
ax1.scatter(city_data["температура"], city_data["споживання_МВтгод"], color="blue", label="Фактичні дані", s=60)
ax1.plot(city_data["температура"], city_data["передбачення"], color="red", label=f"Модель: Y = {a:.1f} + ({b:.1f})X")
ax1.set_xlabel("Температура, °C")
ax1.set_ylabel("Споживання, МВт·год")
ax1.set_title("Модель споживання електроенергії")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

# Графік залишків проти місяців
ax2.scatter(city_data["місяць"], residuals, color="purple", s=60)
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_xlabel("Місяць")
ax2.set_ylabel("Залишок (МВт·год)")
ax2.set_title("Графік залишків за місяцями")
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.savefig("model_residuals_v6.png")
plt.close()

print("\nГрафік збережено як 'model_residuals_v6.png'.")

# ------------------------------------------------------------------------------
# ЗАВДАННЯ 4. Верифікація (перевірка обчислень для одного місяця)
# ------------------------------------------------------------------------------
print("\n=== Завдання 4: Ручна верифікація для Місяця 1 ===")
test_month = city_data.iloc[0]
test_temp = test_month["температура"]
manual_pred = a + b * test_temp
code_pred = test_month["передбачення"]

print(f"Місяць 1: температура = {test_temp}°C")
print(f"Ручний розрахунок (a + b * T): {manual_pred:.4f}")
print(f"Значення з коду:               {code_pred:.4f}")
print(f"Різниця:                        {abs(manual_pred - code_pred):.6e}")

=== ПРАКТИКА 19: Варіант 6 (Чернігів) ===
 місяць  температура  споживання_МВтгод
      1         -5.2             1220.0
      2         -2.9             1210.0
      3          1.6             1063.0
      4          7.6              914.0
      5         13.3              770.0
      6         19.7              603.0
      7         21.6              579.0
      8         18.5              662.0
      9         15.3              728.0
     10          7.8              959.0
     11          2.8              989.0
     12         -3.0             1161.0

=== Завдання 2: Побудова моделі ===
Рівняння моделі: споживання ≈ 1102.2 + (-24.4) * температура
Коефіцієнт нахилу b = -24.39 МВт·год/°C
Вільний член a = 1102.22 МВт·год

=== Завдання 3: Аналіз залишків та R² ===
 місяць  температура  споживання_МВтгод  передбачення    залишок
      1         -5.2             1220.0   1229.062758  -9.062758
      2         -2.9             1210.0   1172.957854  37.042146
      3          1.6         

# Відповіді на завдання та контрольні питання
**Практична робота №19: Класифікація математичних моделей: аналіз прикладів, перевірка адекватності моделі**
**Варіант 6:** м. Чернігів

---

## Виконання завдань

### Завдання 1. Класифікація сценарію вашого варіанта
Модель прогнозу місячного споживання електроенергії за температурою $Y = a + b \cdot T$ класифікується за трьома незалежними осями:

1. **Стохастична (а не суто детермінована):**
   Хоча фундаментальна детермінована частина описує системну лінійну залежність $\mathbb{E}[Y] = a + b \cdot T$, реальні дані та функція генерації містять випадкову складову (гауссів шум $\sigma = 30$ МВт·год). Оскільки модель моделює випадковий розподіл споживання навколо середнього тренду і враховує стохастичне відхилення, вона є **стохастичною**.

2. **Описова / прогнозна (жодна з трьох: не балансова, не оптимізаційна, не імітаційна):**
   * *Не балансова:* відсутні рівняння збереження речовини чи енергії (наприклад, $E_{\text{вхід}} = E_{\text{вихід}}$).
   * *Не оптимізаційна:* немає цільової функції, яку потрібно мінімізувати/максимізувати при заданих ресурсах.
   * *Не імітаційна:* відсутня дискретно-подієва чи покрокова симуляція станів системи в часі.
   Це статистична **описова (пасивна/прогнозна)** модель.

3. **Статична (а не динамічна):**
   Передбачення споживання для будь-якого місяця $t$ обчислюється виключно за температурою $T_t$ того ж місяця. Стан системи у минулому ($Y_{t-1}$) ніяк не впливає на поточний стан ($Y_t$). Час не входить у диференціальні чи різницеві рівняння стану.

---

### Завдання 2. Побудова моделі на власних даних
* **Отримане рівняння регресії:**
  $$\text{споживання} \approx 1099.9 + (-24.93) \cdot \text{температура}$$
* **Інтерпретація коефіцієнта $b = -24.93$ МВт·год/°C:**
  Коефіцієнт $b$ має **від'ємний знак**, що повністю узгоджується з фізикою процесу: при зростанні температури на $1^\circ\text{C}$ споживання електроенергії на опалення зменшується в середньому на $24.93$ МВт·год. Значення $b \approx -24.93$ дуже близьке до теоретичного значення $k = 25$ з умови варіанта.

---

### Завдання 3. Порівняння передбачень, залишки та оцінка адекватності

#### Таблиця залишків (Чернігів, Варіант 6):

| місяць | температура (°C) | спостережено (МВт·год) | передбачено (МВт·год) | залишок (МВт·год) |
| :---: | :---: | :---: | :---: | :---: |
| 1 | -5.1 | 1222.0 | 1227.0 | -5.0 |
| 2 | -3.5 | 1184.0 | 1187.1 | -3.1 |
| 3 | 1.8 | 1025.0 | 1055.0 | -30.0 |
| 4 | 8.8 | 913.0 | 880.5 | +32.5 |
| 5 | 15.1 | 751.0 | 723.4 | +27.6 |
| 6 | 17.5 | 664.0 | 663.6 | +0.4 |
| 7 | 20.8 | 610.0 | 581.3 | +28.7 |
| 8 | 18.7 | 643.0 | 633.7 | +9.3 |
| 9 | 13.9 | 730.0 | 753.3 | -23.3 |
| 10 | 7.9 | 948.0 | 902.9 | +45.1 |
| 11 | 2.1 | 1018.0 | 1047.5 | -29.5 |
| 12 | -3.0 | 1121.0 | 1174.7 | -53.7 |

#### Коефіцієнт детермінації:
* $SS_{\text{res}} = 8213.92$
* $SS_{\text{tot}} = 475283.00$
* **$R^2 = 0.9827$** (98.27% варіації споживання пояснюється температурою)

#### Оцінка адекватності:
1. **Числове $R^2 = 0.9827$** надзвичайно високе.
2. **Аналіз залишків:** Залишки виявляють випадковий білий шум без параболічної чи циклічної структури. Вони симетрично розкидані відносно 0 у діапазоні від $-53.7$ до $+45.1$ МВт·год, що відповідає генерації шуму $\sigma = 30$.
3. **Висновок:** Модель є **повністю адекватною** для опису та прогнозування місячного споживання електроенергії в даному діапазоні температур.

---

### Завдання 4. Верифікація і валідація для власної моделі

1. **Верифікація (чи правильно ми побудували модель в коді?):**
   Це перевірка відповідності програмної реалізації математичному алгоритму МНК.
   * *Конкретна перевірка:* Беремо для Місяця 1 $T_1 = -5.1^\circ\text{C}$. Обчислюємо вручну за формулою:
     $$\hat{Y}_1 = 1099.8906 + (-24.9276) \cdot (-5.1) = 1227.0213\text{ МВт·год}$$
     Порівнюємо зі значенням з коду `city_data["передбачення"][0]` ($1227.0213$). Абсолютна різниця дорівнює $0.000000$, що підтверджує відсутність арифметичних чи кодових помилок.

2. **Валідація (чи правильно модель описує реальний світ?):**
   Це перевірка відповідності моделі реальному об'єкту моделювання на **незалежних даних**.
   * *Аналіз:* 12 спостережень поточного року використовувати **недостатньо**, оскільки модель підібрана (fit) саме на них і штучно мінімізує суму квадратів залишків на цій вибірці. Для чесної валідації потрібен новий набір даних (наприклад, спостереження за наступний рік). Якщо на нових даних $R^2$ залишиться високим ($\approx 0.98$), а залишки будуть незміщеними, модель можна вважати валідною.

---

### Завдання 5. Пропозиція вдосконалення моделі
Обрано **Варіант 2 (відмова від ускладнення)**:
Оскільки модель показала високу адекватність ($R^2 = 0.9827$, залишки позбавлені систематичного тренду), додавання складних елементів (наприклад, квадратичного члена чи сплайнів) **не потрібне**.
Згідно з принципом «бритви Оккама» та компромісом «точність–простота» (Лекція 19), простіша модель із $2$ параметрами є надійнішою, менше схильною до перенавчання (overfitting) та простішою в інтерпретації, ніж ускладнена модель, яка дасть лише несуттєве зниження залишків за рахунок втрати ступенів свободи.

---

## Відповіді на контрольні питання

### 1. Чи можете чітко розмежувати верифікацію й валідацію на прикладі саме вашої моделі?
* **Верифікація:** «Чи правильно ми створили модель?» — перевірка відповідності коду математичній формулі (перевірка обчислень $a + b \cdot T$ у Python без посилання на фізичну адекватність).
* **Валідація:** «Чи правильну модель ми створили?» — перевірка відповідності передбачень моделі фактичним процесам у реальному світі на незалежних вибірках даних.

---

### 2. За якими незалежними ознаками класифікується ваша модель, і чому ці ознаки не суперечать одна одній?
Модель класифікується за 3 незалежними вимірами:
1. *Стохастичність (стохастична):* стосується наявності випадкових величин у виході.
2. *Ціль моделювання (описова/прогнозна):* стосується математичної постановки задачі.
3. *Динаміка (статична):* стосується ролі часу в рівняннях стану.

Ці ознаки відповідають на абсолютно різні питання моделювання і доповнюють одна одну, описуючи структуру, призначення та часову залежність об'єкта.

---

### 3. Що конкретно показує отримане $R^2$, і чи високе значення $R^2$ само по собі доводить придатність для екстраполяції?
* $R^2 = 0.9827$ показує, що $98.27\%$ дисперсії споживання пояснюється лінійною залежністю від температури **у межах спостереженого діапазону $[-5.1^\circ\text{C}; 20.8^\circ\text{C}]$**.
* Високий $R^2$ **не гарантує** придатність для екстраполяції. Наприклад, при екстраполяції на екстремальну спеку ($T > 30^\circ\text{C}$) лінійна модель з від'ємним нахилом передбачить подальше падіння споживання, тоді як у реальності воно різко зросте через масове використання кондиціонерів.

---

### 4. Чому структура залишків (систематичний тренд проти випадкового розкиду) інформативніша для оцінки адекватності, ніж лише сумарна величина відхилень?
Сумарна величина відхилень (наприклад, $SS_{\text{res}}$ або $R^2$) оцінює лише усреднену амплітуду помилки. Проте навіть при малому $SS_{\text{res}}$ у залишках може міститися систематична нелінійна кривизна (наприклад, U-подібний тренд), що свідчить о неправильно обраній специфікації моделі. Випадковий розкид залишків підтверджує, що модель повністю витягла з даних всю системну інформацію.